# UAS KAB - Eksperimen Forecasting dan Rekomendasi Portofolio Saham

**Nama:** Aisya Ulul Asmi  
**NIM:** 25/564969/PPA/07123  
**Mata Kuliah:** Kecerdasan Artifisial pada Bisnis  

Notebook ini mendokumentasikan alur eksperimen mulai dari pemuatan dataset, feature engineering, proses modeling/forecasting, evaluasi model, ranking saham, alokasi portofolio, sampai visualisasi output. Notebook ini disiapkan sebagai dokumentasi kode eksperimen yang dapat dibaca dan dijalankan ulang secara bertahap.

## Ringkasan Alur Eksperimen

Tahapan yang dilakukan:

1. Memuat dataset harga saham dari folder `data/raw`.
2. Menggabungkan data saham menjadi satu dataset historis.
3. Membuat fitur teknikal untuk kebutuhan model machine learning.
4. Menjalankan pipeline forecasting utama menggunakan TimesFM, RandomForest, dan CatBoost.
5. Membaca hasil evaluasi model, forecast, ranking saham, rekomendasi BUY/HOLD, dan alokasi portofolio.
6. Menampilkan visualisasi hasil forecast dan portofolio.

Catatan: proses training/forecasting penuh dapat memakan waktu dan bergantung pada ketersediaan dependency seperti `catboost`, `timesfm`, dan `torch`. Jika checkpoint TimesFM tidak tersedia secara lokal, pipeline memakai fallback forecasting yang sudah terdokumentasi di kode.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))

DATA_DIR = ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = ROOT / "outputs"
TABLES_DIR = OUTPUTS_DIR / "tables"
FIGURES_DIR = OUTPUTS_DIR / "figures"

print("Project root:", ROOT)
print("Raw data dir:", RAW_DIR)
print("Output tables dir:", TABLES_DIR)
print("Output figures dir:", FIGURES_DIR)

## 1. Konfigurasi Eksperimen

Bagian ini membaca konfigurasi utama dari modul project. Konfigurasi mencakup daftar saham, horizon forecast, modal simulasi, seed, serta daftar fitur teknikal yang digunakan model.

In [ ]:
from src.config import (
    TICKERS,
    COMPANY_NAMES,
    SECTORS,
    START_DATE,
    HORIZON_DAYS,
    TOTAL_CAPITAL_IDR,
    RANDOM_STATE,
    FEATURE_COLUMNS,
)

print("Ticker yang dianalisis:", TICKERS)
print("Start date:", START_DATE)
print("Horizon forecast:", HORIZON_DAYS, "trading days")
print("Modal simulasi: Rp{:,.0f}".format(TOTAL_CAPITAL_IDR))
print("Random state:", RANDOM_STATE)
print("Jumlah fitur teknikal:", len(FEATURE_COLUMNS))

## 2. Memuat Dataset Saham

Dataset utama berada pada folder `data/raw`. Setiap ticker memiliki file CSV sendiri, kemudian seluruh data digabungkan untuk kebutuhan analisis dan feature engineering.

In [ ]:
raw_files = sorted(RAW_DIR.glob("*.csv"))
print("Jumlah file raw CSV:", len(raw_files))
for file in raw_files:
    print("-", file.name)

raw_frames = []
for file in raw_files:
    df = pd.read_csv(file)
    if "Ticker" not in df.columns:
        df["Ticker"] = file.stem
    raw_frames.append(df)

all_stocks = pd.concat(raw_frames, ignore_index=True)
all_stocks["Date"] = pd.to_datetime(all_stocks["Date"])
all_stocks = all_stocks.sort_values(["Ticker", "Date"]).reset_index(drop=True)

display(all_stocks.head())
display(all_stocks.tail())
print("Shape dataset gabungan:", all_stocks.shape)

In [ ]:
dataset_summary = (
    all_stocks.groupby("Ticker")
    .agg(
        start_date=("Date", "min"),
        end_date=("Date", "max"),
        rows=("Date", "count"),
        first_close=("Close", "first"),
        last_close=("Close", "last"),
        avg_volume=("Volume", "mean"),
    )
    .reset_index()
)
dataset_summary["historical_return"] = dataset_summary["last_close"] / dataset_summary["first_close"] - 1
display(dataset_summary)

## 3. Feature Engineering

Fitur teknikal dibentuk dari harga dan volume historis. Fitur yang digunakan antara lain return historis, moving average, exponential moving average, RSI, MACD, stochastic oscillator, volatility, ATR, ADX, volume indicators, dan posisi harga terhadap rentang 52 minggu.

In [ ]:
from src.features import make_features_one, make_modeling_dataset

feature_sample = make_features_one(all_stocks[all_stocks["Ticker"] == TICKERS[0]])
print("Contoh feature untuk", TICKERS[0])
display(feature_sample[["Date", "Ticker", "Close"] + FEATURE_COLUMNS[:10]].tail())

modeling_dataset = make_modeling_dataset(all_stocks)
print("Shape modeling dataset:", modeling_dataset.shape)
display(modeling_dataset.head())

## 4. Menjalankan Pipeline Modeling dan Forecasting

Pipeline utama berada pada `main.py`. Pipeline ini menjalankan:

- pemuatan dan penggabungan data,
- feature engineering,
- training RandomForest dan CatBoost,
- forecasting TimesFM/fallback,
- ensemble forecast,
- walk-forward validation,
- ranking saham,
- alokasi portofolio,
- penyimpanan tabel dan gambar output.

Secara default, cell di bawah tidak menjalankan ulang pipeline agar notebook tidak memakan waktu lama. Ubah `RUN_FULL_PIPELINE = True` jika ingin menghasilkan ulang seluruh output dari awal.

In [ ]:
RUN_FULL_PIPELINE = False

if RUN_FULL_PIPELINE:
    from main import main
    main()
else:
    print("Pipeline penuh tidak dijalankan ulang. Notebook akan membaca output yang sudah tersedia di folder outputs/.")

## 5. Membaca Output Eksperimen

Bagian ini membaca tabel hasil eksperimen dari `outputs/tables`. Tabel ini merupakan hasil utama yang juga dipakai oleh dashboard Streamlit dan laporan.

In [ ]:
def read_table(name: str) -> pd.DataFrame:
    path = TABLES_DIR / name
    if not path.exists():
        print(f"File tidak ditemukan: {path}")
        return pd.DataFrame()
    return pd.read_csv(path)

model_metrics = read_table("model_metrics.csv")
forecast_summary = read_table("forecast_summary_6m.csv")
stock_ranking = read_table("stock_ranking.csv")
portfolio = read_table("portfolio_allocation.csv")
portfolio_aggressive = read_table("portfolio_allocation_aggressive.csv")
final_recommendation = read_table("final_recommendation.csv")
validation_summary = read_table("model_validation_summary.csv")
risk_metrics = read_table("risk_metrics.csv")
scenario_summary = read_table("scenario_summary.csv")
confidence = read_table("recommendation_confidence_levels.csv")

print("Tabel output berhasil dibaca.")

## 6. Evaluasi Model

Evaluasi model menggunakan error metrics seperti MAE, RMSE, MAPE, serta DSTAT untuk mengukur ketepatan arah prediksi. DSTAT menunjukkan kemampuan model membaca arah pergerakan, bukan hanya besaran error harga/return.

In [ ]:
display(model_metrics.head(20))

if not model_metrics.empty:
    metrics_view = model_metrics.rename(columns={"directional_accuracy": "DSTAT"})
    summary_eval = metrics_view.groupby("model").agg(
        avg_mae=("mae", "mean"),
        avg_rmse=("rmse", "mean"),
        avg_mape=("mape", "mean"),
        avg_DSTAT=("DSTAT", "mean"),
    ).reset_index()
    display(summary_eval)

In [ ]:
if not model_metrics.empty:
    plot_df = model_metrics.rename(columns={"directional_accuracy": "DSTAT"})
    fig, ax = plt.subplots(figsize=(10, 5))
    plot_df.groupby("model")["mape"].mean().sort_values().plot(kind="bar", ax=ax, color="#1f6feb")
    ax.set_title("Rata-rata MAPE per Model")
    ax.set_ylabel("MAPE")
    ax.set_xlabel("Model")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


## 7. Hasil Forecast 6 Bulan

Tabel forecast 6 bulan berisi harga terakhir, prediksi harga, expected return, kontribusi masing-masing model, probabilitas BUY, dan konsistensi forecast.

In [ ]:
display(forecast_summary)

if not forecast_summary.empty and "Ensemble_Return_6M" in forecast_summary.columns:
    fig, ax = plt.subplots(figsize=(11, 5))
    ordered = forecast_summary.sort_values("Ensemble_Return_6M", ascending=False)
    ax.bar(ordered["Ticker"], ordered["Ensemble_Return_6M"], color="#0f766e")
    ax.set_title("Expected Return 6 Bulan Berdasarkan Ensemble Forecast")
    ax.set_ylabel("Expected Return 6M")
    ax.set_xlabel("Ticker")
    ax.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.show()


## 8. Ranking Saham dan Rekomendasi BUY/HOLD

Ranking saham mempertimbangkan expected return, excess return terhadap benchmark internal, DSTAT, probabilitas BUY, likuiditas, volatilitas, drawdown, posisi harga, dan skor risiko. Saham dengan skor terbaik dipilih sebagai kandidat portofolio.

In [ ]:
ranking_view = stock_ranking.copy()
ranking_view = ranking_view.rename(columns={
    "directional_accuracy": "DSTAT",
    "norm_directional_accuracy": "norm_DSTAT",
})
display(ranking_view.head(10))

print("Rekomendasi final:")
display(final_recommendation)

print("Confidence level:")
display(confidence)

## 9. Alokasi Portofolio

Alokasi portofolio disusun dalam beberapa skenario. Skenario balanced digunakan sebagai rekomendasi utama, sedangkan skenario conservative dan aggressive digunakan sebagai pembanding sensitivitas terhadap risiko dan konsentrasi bobot.

In [ ]:
display(portfolio)
display(portfolio_aggressive)
display(scenario_summary)

if not portfolio.empty and {"ticker", "final_weight"}.issubset(portfolio.columns):
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.pie(portfolio["final_weight"], labels=portfolio["ticker"], autopct="%1.1f%%", startangle=90)
    ax.set_title("Alokasi Portofolio Balanced")
    plt.tight_layout()
    plt.show()


## 10. Analisis Risiko dan Validasi

Bagian ini menampilkan ringkasan risiko dan walk-forward validation. Validasi walk-forward penting karena model forecasting harus diuji secara kronologis, bukan dengan split acak, agar tidak terjadi kebocoran informasi dari masa depan.

In [ ]:
display(risk_metrics)
display(validation_summary.rename(columns={
    "avg_directional_accuracy": "avg_DSTAT"
}))


## 11. Visualisasi Forecast dan Portofolio

Gambar visualisasi diambil dari folder `outputs/figures`. Visualisasi ini sama dengan yang ditampilkan pada dashboard Streamlit.

In [ ]:
def show_png(path: Path, width: int = 1100):
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print("File tidak ditemukan:", path)

selected_ticker = "AAPL"
show_png(FIGURES_DIR / f"forecast_panel_{selected_ticker}.png")
show_png(FIGURES_DIR / "forecast" / f"{selected_ticker}_forecast.png", width=850)
show_png(FIGURES_DIR / "price_history" / f"{selected_ticker}_price_history.png", width=850)
show_png(FIGURES_DIR / "portfolio" / "portfolio_allocation.png", width=800)

## 12. Kesimpulan Eksperimen

Eksperimen menghasilkan rekomendasi portofolio berdasarkan kombinasi forecasting time series, model tabular machine learning, evaluasi arah prediksi, risk scoring, dan simulasi alokasi modal. Output utama berupa tabel evaluasi, ranking saham, rekomendasi final, visualisasi forecast, dan alokasi portofolio tersimpan di folder `outputs/` serta digunakan kembali oleh dashboard Streamlit.

Untuk menjalankan ulang hasil penuh, gunakan `RUN_FULL_PIPELINE = True` pada bagian pipeline atau jalankan `python main.py` dari root project. Untuk membuka dashboard, jalankan `streamlit run streamlit_app.py` pada folder deploy atau `streamlit run app/streamlit_app.py` pada struktur project lengkap.